# Tome 14, Chapter 11: Gaps and clumps (Python companion)

Exercises first, solutions at the end. We compute Buchstab's function and its sign changes, compare it with actual counts of integers free of small primes, build a Maier matrix, run the Erdős–Rankin covering construction and produce an explicit long run of composite numbers. The computer science application: **the bias of incremental prime generation**, the common recipe "pick a random odd number and step forward until you hit a prime", which favours primes that follow long gaps. Requires only `numpy`.

In [ ]:
import math, time
import numpy as np

def small_primes(n):
    s = np.ones(n + 1, dtype=bool); s[:2] = False
    for i in range(2, int(n ** 0.5) + 1):
        if s[i]: s[i * i::i] = False
    return np.nonzero(s)[0]

def is_prime(n):
    # deterministic Miller-Rabin for n < 3.3e24
    if n < 2: return False
    small = (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41)
    for p in small:
        if n % p == 0: return n == p
    d, s = n - 1, 0
    while d % 2 == 0: d //= 2; s += 1
    for a in small:
        y = pow(a, d, n)
        if y in (1, n - 1): continue
        for _ in range(s - 1):
            y = y * y % n
            if y == n - 1: break
        else: return False
    return True

## Exercises

**P11.1 (★★) Buchstab's function.** Compute $\omega(u)$ on $[1,9]$ from $u\omega(u)=1+\int_1^{u-1}\omega$ with a step of $10^{-5}$. Locate the sign changes of $\omega(u)-e^{-\gamma}$ and the extreme values of $e^\gamma\omega(u)$ after $u=2$. Then, for $x=10^8$ and $u=1.5,2,2.5,3,4$, compare $\Phi(x,x^{1/u})\frac{\log z}x$ with $\omega(u)$ (Theorem 11.3).

**P11.2 (★★) A Maier matrix.** With $P=510510$, the product of the primes up to $17$, count the primes in $]rP,rP+y]$ for $1000\le r<2000$ and $y=17^u$, $1\le u\le3.5$. Compare the average, normalised by $\frac y{\log rP}$, with the prediction $\frac P{\varphi(P)}\frac{\Phi_P(y)}y$ of the proof of Theorem 11.6, and with $1000$ random starting points (Figure 11.2).

**P11.3 (★★) The Erdős–Rankin covering.** For $x=50,100,200,400$ and small $z$, cover $[1,y]$ by the classes $1\bmod p$ ($p\le z$), $0\bmod p$ ($z<p\le\frac x2$), and for each prime of $]\frac x2,x]$ the class that covers the most remaining integers (greedy). Find the largest $y$ that the greedy covering reaches, compare with the naive $\frac x2$, and, for $x=50$, build the integer $n$ of Lemma 11.8 with the Chinese remainder theorem and check that $n+1,\dots,n+y$ are composite. How long is the actual prime gap containing them?

**P11.4 (★, CS application) Biased prime generation.** Many libraries generate a random prime by choosing a random odd $m$ in a range and returning the first prime $\ge m$. A prime $p$ is then returned with probability proportional to the gap $g(p)=p-p^-$ to the previous prime. For the primes in $[10^{12},10^{12}+10^7]$, compute the largest and smallest selection probability relative to uniform, and the entropy of the output distribution compared with $\log_2$ of the number of primes. What does Maier's theorem say about where the over-selected primes live?

## Solutions

### P11.1

In [ ]:
def buchstab(U=9, n=100000):
    h = 1.0 / n; N = int((U - 1) * n) + 1
    us = 1 + h * np.arange(N); om = np.empty(N); I = np.zeros(N)   # I = int_1^u omega
    om[:n + 1] = 1 / us[:n + 1]
    I[1:n + 1] = np.cumsum((om[1:n + 1] + om[:n]) * h / 2)
    for b in range(1, U - 1):
        idx = np.arange(b * n + 1, min((b + 1) * n, N - 1) + 1)
        om[idx] = (1 + I[idx - n]) / us[idx]
        seg = om[b * n:idx[-1] + 1]
        I[b * n + 1:idx[-1] + 1] = I[b * n] + np.cumsum((seg[1:] + seg[:-1]) * h / 2)
    return us, om

us, om = buchstab()
G = 0.5772156649015329
d = math.exp(G) * om - 1
m = (us > 1.5) & (us < 7.6)
s = np.sign(d[m]); print("sign changes:", np.round(us[m][1:][s[1:] != s[:-1]], 3))
for a, b in [(2, 3.2), (3.2, 4)]:
    k = (us >= a) & (us <= b)
    i, j = np.argmax(d[k]), np.argmin(d[k])
    print(f"on [{a},{b}]: max e^g omega = {1 + d[k][i]:.5f} at u={us[k][i]:.3f}, min = {1 + d[k][j]:.5f} at u={us[k][j]:.3f}")

In [ ]:
X = 10 ** 8
spf = np.zeros(X + 1, dtype=np.int32)          # smallest prime factor
for p in small_primes(int(X ** 0.5)):
    p = int(p); blk = spf[p::p]; blk[blk == 0] = p
spf[spf == 0] = np.arange(X + 1)[spf == 0]     # primes (and 0, 1)
spf[1] = 10 ** 9                               # 1 has no prime factor
for u in [1.5, 2, 2.5, 3, 4]:
    z = X ** (1 / u)
    Phi = int(np.count_nonzero(spf[1:] >= z))
    print(f"u={u}: Phi*log z/x = {Phi * math.log(z) / X:.4f},  omega(u) = {om[int(round((u - 1) * 100000))]:.4f}")

The sign changes come at $u=1.781$ (that is $e^\gamma$), $2.483$, $3.227$, $4.002$, $4.786$, $5.567$, $6.351$, $7.140$, roughly once per unit, and the amplitude collapses: after $u=2$ the largest value of $e^\gamma\omega$ is $1.0101$ at $u=2.763$ (Exercise 11.1), the next minimum only $0.99887$. So Maier's effect is large for $\lambda<2$ and tiny but real beyond. The counts up to $10^8$ follow $\omega(u)$ within $6\%$ (and within $2\%$ for $u\ge2.5$), which is the $O(\frac1{\log z})$ of Theorem 11.3: $\log z$ is only $6$ to $12$ here.

### P11.2

In [ ]:
P = 510510; phiP = 1 * 2 * 4 * 6 * 10 * 12 * 16
Ymax = int(17 ** 3.5) + 1
sp = small_primes(40000)
def prime_cumcount(lo, L):
    # c[k] = number of primes in ]lo, lo+k]  (lo > 40000^2 / L is not needed: lo is large here)
    s = np.ones(L + 1, dtype=bool); s[0] = False
    for p in sp:
        p = int(p)
        if p * p > lo + L: break
        s[(-lo) % p::p] = False
    return np.cumsum(s)
us2 = np.round(np.arange(1.0, 3.51, 0.25), 2); ys = (17.0 ** us2).astype(int)
k = np.arange(1, Ymax + 1); cop = np.ones(Ymax, dtype=bool)
for p in [2, 3, 5, 7, 11, 13, 17]: cop[k % p == 0] = False
pred = np.cumsum(cop)[ys - 1] * P / (phiP * ys)
rng = np.random.default_rng(11)
obs = np.zeros(len(ys)); rnd = np.zeros(len(ys))
for r in range(1000, 2000):
    c = prime_cumcount(r * P, Ymax); obs += c[ys] / (ys / math.log(r * P))
    lo = int(rng.integers(1000 * P, 2000 * P)); c = prime_cumcount(lo, Ymax); rnd += c[ys] / (ys / math.log(lo))
obs /= 1000; rnd /= 1000
for row in zip(us2, ys, np.round(obs, 3), np.round(pred, 3), np.round(rnd, 3)):
    print("u=%.2f  y=%6d  matrix %.3f  prediction %.3f  random x %.3f" % row)

The rows of the Maier matrix behave exactly as the proof says, column by column: at $u=1.75$ ($y=142$) the intervals starting at multiples of $P$ carry $8\%$ more primes than $\frac y{\log x}$, at $u=1.25$ about $19\%$ fewer, and the observed ratios agree with $\frac P{\varphi(P)}\frac{\Phi_P(y)}y$ to within about $1\%$. Random starting points give ratios within $4\%$ of $1$ at every $u$. With $z=17$ the prediction is not yet $e^\gamma\omega(u)$ (for instance $\frac P{\varphi(P)}=5.21$ while $e^\gamma\log17=5.05$), but the shape, a dip, a peak and a damped oscillation, is the same.

### P11.3

In [ ]:
def greedy_cover(x, z, y):
    ps = [int(p) for p in small_primes(x)]
    alive = np.ones(y + 1, dtype=bool); alive[0] = False
    a = {}
    for p in ps:
        if p <= z: a[p] = 1
        elif p <= x // 2: a[p] = 0
        else: continue
        alive[(a[p] or p)::p] = False
    for p in ps:
        if p <= x // 2: continue
        idx = np.nonzero(alive)[0]
        r = int(np.argmax(np.bincount(idx % p, minlength=p))) if len(idx) else 0
        a[p] = r; alive[(r or p)::p] = False
    return not alive.any(), a

def best_y(x):
    best = (0, None)
    for z in [1, 2, 3, 5, 7, 11, 13]:
        y = x // 2
        while greedy_cover(x, z, y + 1)[0]: y += 1
        if y > best[0]: best = (y, z)
    return best
for x in [50, 100, 200, 400]:
    y, z = best_y(x)
    print(f"x={x}: greedy covers [1,{y}] (z={z}),  y/x = {y / x:.2f},  naive x/2 = {x // 2}")

In [ ]:
x = 50; y, z = best_y(x); ok, a = greedy_cover(x, z, y)
ps = [int(p) for p in small_primes(x)]
P = math.prod(ps); n = 0
for p in ps:                                    # CRT: n = -a_p mod p
    M = P // p; n = (n + (-a[p] % p) * M * pow(M, -1, p)) % P
n += P                                          # make n > x
print("all composite:", all(not is_prime(n + k) for k in range(1, y + 1)))
lo = n; hi = n + y + 1
while not is_prime(lo): lo -= 1
while not is_prime(hi): hi += 1
print(f"n has {len(str(n))} digits; the prime gap around the run: {lo} .. {hi}, length {hi - lo}; log n = {math.log(n):.1f}, log^2 n = {math.log(n) ** 2:.0f}")

The naive covering ($a_p=0$ throughout) reaches only $y\approx\frac x2$. Using the class $1$ for the primes $2$ (and sometimes $3$) and letting each large prime cover several survivors (the greedy step, a baby version of the idea of Ford, Green, Konyagin, Maynard and Tao) already reaches $y\approx1.5x$ to $2x$. Rankin's extra factor $\frac{\log x\log_3x}{(\log_2x)^2}$ is invisible at this scale, which is typical of iterated logarithms. The explicit run of composites lies far above anything we could find by search: $n$ has $18$ digits. The covering guarantees $73$ composites; the actual prime gap around them has length $220$, about $5.4$ times the average gap $\log n=41$, and the covering is what tells us where to look.

### P11.4

In [ ]:
lo, L = 10 ** 12, 10 ** 7
s = np.ones(L, dtype=bool)
for p in small_primes(int(math.isqrt(lo + L)) + 1):
    p = int(p); s[(-lo) % p::p] = False
pr = lo + np.nonzero(s)[0]
gaps = np.diff(pr)                              # gap before pr[1:]
prob = gaps / gaps.sum()                        # selection probability of pr[1:]
Np = len(gaps)
H = -np.sum(prob * np.log2(prob))
print(f"{Np} primes, mean gap {gaps.mean():.2f} (log x = {math.log(lo):.2f})")
print(f"max / uniform = {prob.max() * Np:.2f}, min / uniform = {prob.min() * Np:.3f}")
print(f"entropy {H:.4f} bits vs log2(#primes) = {math.log2(Np):.4f} bits: loss {math.log2(Np) - H:.4f} bits")
top = np.argsort(prob)[-5:]
print("most likely outputs (prime, gap before it):", [(int(pr[1:][i]), int(gaps[i])) for i in top])

The incremental method returns the most favoured prime (the one after the longest gap in the window) about twelve times more often than a uniform choice would, and the twin-prime second members, after a gap of $2$, about fourteen times less often. Yet the entropy loss is small, about half a bit, because $\mathbb E[g\log g]/\mathbb E[g]-\log\mathbb E[g]$ is a constant when the gaps are roughly exponential. That is the Cramér model at work. Maier's theorem adds a warning: the primes after long gaps cluster where the small primes conspire, for instance just after multiples of products of small primes, so the bias is not spread evenly, and it is arithmetic rather than random.